# MomBased_Pt3 — Refined Dow-30 Momentum

**What changed vs Pt2 (at a glance):**

| Area | Pt2 | Pt3 |
|---|---|---|
| Features (predictors) | 3: `mom_12_1`, `vol_60`, `rsi` | 8: `mom_12_1`, `mom_6_1`, `mom_3_1`, `mom_accel`, `mom_21`, `ma_cross`, `dist_52wk_high`, `rsi` |
| Target | Raw 5-day fwd return → rank | **Risk-adjusted** 5-day fwd return `(fwd / max(vol_60, 5%))` → rank |
| Regime thresholds | Hardcoded magic numbers | Calibrated to **train-sample percentiles** |
| Optuna trials | 5 | **150** |
| CV inside Optuna | None (single-fold val Sharpe) | **Purged walk-forward CV, 3 folds**, embargo gap ≥ 5 days |
| Metrics reported | CAGR, Sharpe, max DD, kills | + Information Ratio, Sortino, Calmar, hit rate, turnover, YoY vs DIA |

All other architectural constraints held exactly as in PROMPT2.md: long-only Dow-30 momentum, XGBoost ranker → ridge-penalised weighter → regime-gated exposure, point-in-time universe, no look-ahead.


In [1]:
!pip install xgboost optuna --quiet

In [2]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

import yfinance as yf
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")
np.random.seed(20160101)
os.makedirs("results", exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Libs loaded. Seed fixed at 20160101.")

Libs loaded. Seed fixed at 20160101.


## 1. Data collection

**What this does.** Pulls 10 years of daily auto-adjusted Dow-30 closes from Yahoo Finance and reconstructs the point-in-time (PIT) membership set from a 2016 baseline plus a hand-coded list of index changes. PIT masking prevents survivorship bias — the backtest can only see stocks that were actually in the index on each historical date.

**What changed vs Pt2.** Nothing — same `_BASELINE_APR2016` and `_CHANGES`, same `fetch_prices` behaviour, same WBA/UTX exclusion. This section is a faithful copy of Pt2's data layer so the fairest possible comparison can be made downstream.

In [3]:
_BASELINE_APR2016 = {
    "AAPL","AXP","BA","CAT","CSCO","CVX","DD","DIS","GE","GS","HD","IBM","INTC",
    "JNJ","JPM","KO","MCD","MMM","MRK","MSFT","NKE","PFE","PG","RTX","TRV",
    "UNH","V","VZ","WMT","XOM",
}
_CHANGES = [
    ("2018-06-26", ["WBA"],              ["GE"]),
    ("2019-04-02", ["DOW"],              ["DD"]),
    ("2020-04-06", ["RTX"],              ["UTX"]),
    ("2020-08-31", ["AMGN","CRM","HON"], ["XOM","PFE","RTX"]),
    ("2024-02-26", ["AMZN","SHW"],       ["WBA","INTC"]),
    ("2024-11-01", ["NVDA"],             ["DOW"]),
]

def fetch_prices(start_date="2016-04-01", end_date="2026-04-18"):
    all_tickers = set(_BASELINE_APR2016)
    for _, added, _ in _CHANGES:
        all_tickers.update(added)
    all_tickers -= {"WBA", "UTX"}
    raw = yf.download(sorted(all_tickers), start=start_date, end=end_date,
                      auto_adjust=True, progress=False)
    prices = raw["Close"].ffill(limit=10)
    return prices.dropna(axis=1, how="all")

def get_constituents_on_date(date: pd.Timestamp) -> set:
    constituents = set(_BASELINE_APR2016)
    for d_str, added, removed in _CHANGES:
        if date >= pd.Timestamp(d_str):
            constituents.update(added); constituents -= set(removed)
    constituents -= {"WBA", "UTX"}
    return constituents

print("Fetching prices ...")
prices = fetch_prices()
print(f"  {len(prices)} trading days x {len(prices.columns)} tickers")
print(f"  {prices.index[0].date()} -> {prices.index[-1].date()}")

Fetching prices ...


  2526 trading days x 37 tickers
  2016-04-01 -> 2026-04-17


## 2. Feature engineering

**What this does.** Builds eight momentum-flavoured features per stock per day, plus a risk-adjusted 5-day-forward-return *target*. Features are computed vectorised across the whole price panel with `.rolling()` — much faster than Pt2's per-date Python loop.

**What changed vs Pt2 (this is the core fix).**
1. `vol_60` is removed from the predictor set and moved into the *denominator* of the target. This addresses the "wrong-sign vol coefficient" problem that caused Pt2 to systematically pick low-vol defensives (IBM/CSCO/JNJ/KO/MMM/DIS) and miss the winners (MSFT/AAPL/UNH/V/HD).
2. Six new momentum features replace the missing volatility predictor: `mom_6_1`, `mom_3_1`, `mom_accel = mom_3_1 - mom_12_1`, `mom_21` (short-horizon), `ma_cross = SMA50/SMA200 - 1`, `dist_52wk_high = price / trailing_252d_max - 1`. Richer vocabulary means no single feature dominates.
3. The forward-return target is now `fwd_ret / max(ex_ante_vol_60, 0.05)` — dividing by ex-ante volatility turns the signal into a risk-adjusted alpha rather than raw return, so high-vol winners no longer get penalised just for being volatile.

In [4]:
FEATURE_COLS = [
    "mom_12_1", "mom_6_1", "mom_3_1", "mom_accel",
    "mom_21",   "ma_cross", "dist_52wk_high", "rsi",
]
VOL_FLOOR = 0.05   # 5% ann. vol floor on the risk-adjustment denominator

def _rsi_series(px: pd.Series, period: int) -> pd.Series:
    delta  = px.diff()
    gain   = delta.clip(lower=0).rolling(period).mean()
    loss   = (-delta.clip(upper=0)).rolling(period).mean()
    rs     = gain / loss.replace(0, np.nan)
    out    = 100 - 100 / (1 + rs)
    return out.fillna(100.0)

def compute_features(prices: pd.DataFrame, rsi_period: int = 14,
                     fwd_days: int = 5) -> pd.DataFrame:
    log_ret = np.log(prices / prices.shift(1))

    mom_12_1 = prices.shift(21)  / prices.shift(252) - 1.0
    mom_6_1  = prices.shift(21)  / prices.shift(126) - 1.0
    mom_3_1  = prices.shift(21)  / prices.shift(63)  - 1.0
    mom_21   = prices / prices.shift(21) - 1.0
    mom_acc  = mom_3_1 - mom_12_1

    sma50  = prices.rolling(50).mean()
    sma200 = prices.rolling(200).mean()
    ma_cross = sma50 / sma200 - 1.0

    roll_max_252 = prices.rolling(252).max()
    dist_52wk = prices / roll_max_252 - 1.0

    vol_60 = log_ret.rolling(60).std() * np.sqrt(252)

    rsi_panel = pd.concat({c: _rsi_series(prices[c], rsi_period)
                           for c in prices.columns}, axis=1)

    fwd = prices.shift(-fwd_days) / prices - 1.0
    denom = vol_60.clip(lower=VOL_FLOOR)
    fwd_risk_adj = fwd / denom

    # Stack to long format
    feat_panels = {
        "mom_12_1": mom_12_1, "mom_6_1": mom_6_1, "mom_3_1": mom_3_1,
        "mom_accel": mom_acc, "mom_21": mom_21, "ma_cross": ma_cross,
        "dist_52wk_high": dist_52wk, "rsi": rsi_panel,
    }
    long_frames = []
    for name, df in feat_panels.items():
        s = df.stack(future_stack=True); s.index.names = ["date", "ticker"]
        long_frames.append(s.rename(name))
    fwd_s  = fwd.stack(future_stack=True);          fwd_s.index.names = ["date","ticker"]
    fwdra  = fwd_risk_adj.stack(future_stack=True); fwdra.index.names = ["date","ticker"]
    long_frames.append(fwd_s.rename("fwd_ret"))
    long_frames.append(fwdra.rename("fwd_risk_adj"))
    df_long = pd.concat(long_frames, axis=1).reset_index()

    # PIT mask
    members = {d: get_constituents_on_date(d) for d in df_long["date"].unique()}
    df_long["is_member"] = df_long.apply(
        lambda r: r["ticker"] in members[r["date"]], axis=1
    )
    df_long = df_long[df_long["is_member"]].drop(columns="is_member")

    df_long = df_long.replace([np.inf, -np.inf], np.nan)
    df_long = df_long.dropna(subset=FEATURE_COLS + ["fwd_ret", "fwd_risk_adj"])

    # Cross-sectional rank on the risk-adjusted target
    df_long["fwd_rank"] = df_long.groupby("date")["fwd_risk_adj"].rank(pct=True)
    return df_long

print("Computing features (RSI period = 14 for this preview) ...")
feat_df_14 = compute_features(prices, rsi_period=14)
print(f"  feature rows: {len(feat_df_14):,}  unique dates: {feat_df_14['date'].nunique():,}")
print("  feature stats:")
print(feat_df_14[FEATURE_COLS].describe().T[["mean","std","min","max"]].round(3))

Computing features (RSI period = 14 for this preview) ...


  feature rows: 66,402  unique dates: 2,269
  feature stats:
                  mean     std    min     max
mom_12_1         0.136   0.246 -0.741   2.201
mom_6_1          0.059   0.157 -0.732   1.201
mom_3_1          0.024   0.103 -0.697   0.722
mom_accel       -0.112   0.224 -1.897   1.041
mom_21           0.011   0.075 -0.717   0.792
ma_cross         0.034   0.088 -0.532   0.326
dist_52wk_high  -0.101   0.103 -0.754   0.000
rsi             53.119  17.197  1.275  98.836


## 3. Regime detection

**What this does.** Classifies each trading day into `trending` / `volatile` / `crash` using rolling 20-day annualised volatility and 60-day drawdown on an equal-weight PIT index, then scales exposure: 100% in trending, 70% in volatile, 0% in crash.

**What changed vs Pt2.** Thresholds are no longer the hardcoded 0.17 / 0.28 / -0.07 / -0.13 magic numbers. They are now calibrated as **train-sample percentiles** (75th/95th on vol, 25th/5th on drawdown), so the regime labels have a consistent frequency interpretation instead of fitting a particular sample by accident.

In [5]:
TRAIN_CUTOFF = pd.Timestamp("2020-01-01")
VAL_CUTOFF   = pd.Timestamp("2022-01-01")

def build_market_index(prices: pd.DataFrame) -> pd.Series:
    vals = []
    for date in prices.index:
        mems = get_constituents_on_date(date)
        valid = [t for t in mems if t in prices.columns and pd.notna(prices.loc[date, t])]
        vals.append(prices.loc[date, valid].mean() if valid else np.nan)
    return pd.Series(vals, index=prices.index).ffill()

def calibrate_regime_thresholds(prices: pd.DataFrame,
                                train_end: pd.Timestamp = TRAIN_CUTOFF,
                                vol_window: int = 20, dd_window: int = 60):
    mkt = build_market_index(prices)
    log_r = np.log(mkt / mkt.shift(1))
    roll_vol = log_r.rolling(vol_window).std() * np.sqrt(252)
    roll_pk  = mkt.rolling(dd_window).max()
    roll_dd  = (mkt - roll_pk) / roll_pk

    train_vol = roll_vol.loc[:train_end].dropna()
    train_dd  = roll_dd.loc[:train_end].dropna()

    thresh = dict(
        vol_vol   = float(np.percentile(train_vol, 75)),
        vol_crash = float(np.percentile(train_vol, 95)),
        dd_vol    = float(np.percentile(train_dd, 25)),
        dd_crash  = float(np.percentile(train_dd, 5)),
    )
    return mkt, roll_vol, roll_dd, thresh

def classify_regimes(prices: pd.DataFrame) -> pd.Series:
    mkt, roll_vol, roll_dd, thresh = calibrate_regime_thresholds(prices)
    regimes = pd.Series("trending", index=prices.index, dtype=str)
    regimes[(roll_vol > thresh["vol_vol"])   | (roll_dd < thresh["dd_vol"])]   = "volatile"
    regimes[(roll_vol > thresh["vol_crash"]) | (roll_dd < thresh["dd_crash"])] = "crash"
    return regimes, thresh

regimes, regime_thresh = classify_regimes(prices)
print("Calibrated regime thresholds (train-sample percentiles):")
for k, v in regime_thresh.items():
    print(f"  {k:10s} = {v:+.4f}")
print()
for state in ["trending", "volatile", "crash"]:
    n = int((regimes == state).sum())
    print(f"  {state:10s}: {n:4d} days  ({n/len(regimes)*100:.1f}%)")

Calibrated regime thresholds (train-sample percentiles):
  vol_vol    = +0.1312
  vol_crash  = +0.2405
  dd_vol     = -0.0276
  dd_crash   = -0.0766

  trending  : 1301 days  (51.5%)
  volatile  :  888 days  (35.2%)
  crash     :  337 days  (13.3%)


## 4. XGBoost cross-sectional ranker

**What this does.** Trains a gradient-boosted regression tree ensemble to predict the cross-sectional `fwd_rank` percentile (higher rank = stronger risk-adjusted outperformer). Same engine as Pt2 (XGBoost with `reg:squarederror`), but now with an 8-feature vocabulary so no single feature dominates.

**What changed vs Pt2.** The feature list is the new 8-item set. All other XGBoost hyperparameters are tuneable by Optuna in the next stage.

In [6]:
def train_xgboost(train_df: pd.DataFrame,
                  n_estimators: int = 300,
                  max_depth: int    = 3,
                  learning_rate: float = 0.05,
                  reg_lambda: float = 1.0,
                  reg_alpha: float  = 0.1,
                  subsample: float  = 0.8,
                  min_child_weight: int = 5,
                  random_state: int = 42) -> xgb.XGBRegressor:
    clean = train_df[FEATURE_COLS + ["fwd_rank"]].replace([np.inf,-np.inf],np.nan).dropna()
    X = clean[FEATURE_COLS].values.astype(np.float32)
    y = clean["fwd_rank"].values.astype(np.float32)
    model = xgb.XGBRegressor(
        n_estimators     = n_estimators,
        max_depth        = max_depth,
        learning_rate    = learning_rate,
        subsample        = subsample,
        colsample_bytree = 1.0,
        min_child_weight = min_child_weight,
        reg_lambda       = reg_lambda,
        reg_alpha        = reg_alpha,
        objective        = "reg:squarederror",
        eval_metric      = "rmse",
        random_state     = random_state,
        n_jobs           = -1,
        verbosity        = 0,
        tree_method      = "hist",
    )
    model.fit(X, y)
    return model

def predict_scores(model: xgb.XGBRegressor, feat_df: pd.DataFrame) -> np.ndarray:
    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    return model.predict(X)

## 5. Ridge-penalised long-only weighter

**What this does.** Given XGBoost scores for the PIT universe, pick the top-N and solve `max Σ score_i w_i − λ Σ w_i²` s.t. `Σw=1, w≥0`. Closed-form: `w_i ∝ max(0, score_i) / (2λ)`, then project to the simplex.

**What changed vs Pt2.** Structurally unchanged. λ and N are re-tuned by the expanded Optuna search.

In [7]:
def ridge_optimize(scores: np.ndarray, tickers: list,
                   ridge_lambda: float, top_n: int) -> dict:
    ranked_idx = np.argsort(scores)[::-1][:top_n]
    sel_tickers = [tickers[i] for i in ranked_idx]
    sel_scores  = scores[ranked_idx]
    raw_w = np.maximum(0.0, sel_scores) / (2.0 * ridge_lambda)
    total = raw_w.sum()
    w = raw_w / total if total > 0 else np.full(top_n, 1.0 / top_n)
    return dict(zip(sel_tickers, w))

## 6. Backtest + extended metrics

**What this does.** Walks day-by-day through the chosen evaluation window, rebalances on schedule, scores the PIT universe, allocates with the ridge weighter, scales by regime, and records daily P&L vs a PIT equal-weight benchmark.

**What changed vs Pt2.** Beyond the core CAGR/Sharpe/max-DD/kills returned by Pt2, the backtest now computes:
- **Information ratio (IR)** — excess-return-over-benchmark per unit of tracking error.
- **Sortino ratio** — like Sharpe but only penalises downside volatility.
- **Calmar ratio** — CAGR divided by max drawdown, a return-to-pain measure.
- **Hit rate** — fraction of rebalance-period returns that beat the benchmark.
- **Turnover** — average `Σ|w_i - w_i_prev|` per rebalance (one-sided).
- **Year-by-year** return table for strategy, PIT benchmark, and the **DIA ETF** (true Dow performance).

In [8]:
def run_backtest(prices: pd.DataFrame, features_df: pd.DataFrame,
                 regimes: pd.Series,
                 rebal_freq: int     = 4,
                 ridge_lambda: float = 0.10,
                 top_n: int          = 7,
                 train_end: pd.Timestamp = TRAIN_CUTOFF,
                 eval_start: pd.Timestamp = VAL_CUTOFF,
                 eval_end: pd.Timestamp   = None,
                 xgb_kwargs: dict = None) -> dict:
    rebal_days   = rebal_freq * 5
    xgb_kwargs   = xgb_kwargs or {}
    unique_dates = np.sort(features_df["date"].unique())
    train_dates  = unique_dates[unique_dates < train_end]
    eval_dates   = unique_dates[unique_dates >= eval_start]
    if eval_end is not None:
        eval_dates = eval_dates[eval_dates < eval_end]
    if len(train_dates) < 20 or len(eval_dates) < 20:
        return {"sharpe": -99.0, "cagr": -99.0, "max_dd": -99.0}

    train_df = features_df[features_df["date"].isin(train_dates)]
    model    = train_xgboost(train_df, **xgb_kwargs)
    eval_df  = features_df[features_df["date"].isin(eval_dates)]

    test_start = pd.Timestamp(eval_dates[0])
    test_end   = pd.Timestamp(eval_dates[-1]) if eval_end is None else eval_end
    test_prices = prices[(prices.index >= test_start) & (prices.index <= test_end)].copy()

    weights        = {}
    prev_weights   = {}
    strat_val      = 100.0
    bench_val      = 100.0
    peak           = 100.0
    max_dd         = 0.0
    kills          = 0
    last_rebal     = -rebal_days
    rebal_returns  = []      # strategy return per rebal period
    bench_rebal    = []      # benchmark return per rebal period
    seg_s_start    = strat_val
    seg_b_start    = bench_val
    turnovers      = []

    strat_curve, bench_curve, date_index, regime_log = [], [], [], []

    for di in range(1, len(test_prices)):
        date = test_prices.index[di]
        pit = [t for t in get_constituents_on_date(date) if t in test_prices.columns]

        daily_rets = {}
        for t in pit:
            p0 = test_prices[t].iloc[di-1]; p1 = test_prices[t].iloc[di]
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                daily_rets[t] = p1 / p0 - 1.0

        reg = regimes.get(date, "trending")
        regime_log.append(reg)

        if di - last_rebal >= rebal_days:
            last_rebal = di
            # Close prior rebal segment: record the (strat, bench) returns
            if len(strat_curve) > 0:
                rebal_returns.append(strat_val / seg_s_start - 1.0)
                bench_rebal.append(bench_val   / seg_b_start - 1.0)
                seg_s_start = strat_val
                seg_b_start = bench_val

            avail = eval_df[eval_df["date"] <= date]
            if not avail.empty:
                snap_date = avail["date"].max()
                snap = (eval_df[(eval_df["date"] == snap_date) &
                                (eval_df["ticker"].isin(pit))]
                        .set_index("ticker").dropna(subset=FEATURE_COLS))
                if len(snap) >= top_n:
                    sc   = predict_scores(model, snap.reset_index())
                    tav  = snap.index.tolist()
                    prev_weights = dict(weights)
                    if reg == "crash":
                        weights = {}; kills += 1
                    else:
                        scale = 0.70 if reg == "volatile" else 1.00
                        opt_w = ridge_optimize(sc, tav, ridge_lambda, top_n)
                        weights = {t: w * scale for t, w in opt_w.items()}
                    # Turnover: absolute weight change vs prior rebal
                    all_ts = set(weights) | set(prev_weights)
                    to = sum(abs(weights.get(t,0.0) - prev_weights.get(t,0.0))
                             for t in all_ts)
                    turnovers.append(to)

        strat_ret = sum(weights.get(t,0.0) * daily_rets.get(t,0.0) for t in weights)
        bench_ret = float(np.mean([daily_rets[t] for t in pit if t in daily_rets]))                     if any(t in daily_rets for t in pit) else 0.0
        strat_val *= (1.0 + strat_ret)
        bench_val *= (1.0 + bench_ret)
        peak   = max(peak, strat_val)
        max_dd = min(max_dd, (strat_val - peak) / peak)

        strat_curve.append(strat_val)
        bench_curve.append(bench_val)
        date_index.append(date)

    # close last segment
    if len(strat_curve) > 0:
        rebal_returns.append(strat_val / seg_s_start - 1.0)
        bench_rebal.append(bench_val   / seg_b_start - 1.0)

    if len(strat_curve) < 50:
        return {"sharpe": -99.0, "cagr": -99.0, "max_dd": -99.0}

    strat_arr = np.array(strat_curve); bench_arr = np.array(bench_curve)
    daily_s = np.diff(strat_arr) / strat_arr[:-1]
    daily_b = np.diff(bench_arr) / bench_arr[:-1]
    excess  = daily_s - daily_b

    n_years = len(strat_arr) / 252.0
    cagr       = (strat_val / 100.0) ** (1.0 / n_years) - 1.0
    bench_cagr = (bench_val / 100.0) ** (1.0 / n_years) - 1.0
    sharpe  = (daily_s.mean() / daily_s.std()) * np.sqrt(252) if daily_s.std() > 0 else 0.0

    downside = daily_s[daily_s < 0]
    sortino  = (daily_s.mean() / downside.std()) * np.sqrt(252) if len(downside) > 1 and downside.std() > 0 else 0.0

    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

    ir = (excess.mean() / excess.std()) * np.sqrt(252) if excess.std() > 0 else 0.0

    rr  = np.array(rebal_returns); br = np.array(bench_rebal)
    hit = float(np.mean(rr > br)) if len(rr) > 0 else np.nan

    avg_to = float(np.mean(turnovers)) if turnovers else np.nan

    feat_imp = dict(zip(FEATURE_COLS, model.feature_importances_))

    return {
        "sharpe":        round(float(sharpe), 4),
        "cagr":          round(float(cagr*100), 2),
        "bench_cagr":    round(float(bench_cagr*100), 2),
        "max_dd":        round(float(max_dd*100), 2),
        "cum_ret":       round(strat_val - 100.0, 2),
        "kills":         kills,
        "ir":            round(float(ir), 4),
        "sortino":       round(float(sortino), 4),
        "calmar":        round(float(calmar), 4) if np.isfinite(calmar) else None,
        "hit_rate":      round(float(hit), 4) if np.isfinite(hit) else None,
        "avg_turnover":  round(avg_to, 4) if np.isfinite(avg_to) else None,
        "n_rebals":      len(turnovers),
        "strat_curve":   strat_curve,
        "bench_curve":   bench_curve,
        "date_index":    date_index,
        "regime_log":    regime_log,
        "feat_imp":      feat_imp,
        "model":         model,
        "weights":       weights,
    }

## 7. Optuna with purged walk-forward CV

**What this does.** Bayesian (TPE) search over four hyperparameters — `rsi_period`, `rebal_freq`, `ridge_lambda`, `top_n` — plus two XGBoost regularisation knobs (`reg_lambda`, `max_depth`). The objective evaluates each trial with **purged walk-forward cross-validation**: three folds over the train+val span (2016-04 → 2022-01), each with a 5-day embargo between train-end and val-start to prevent overlapping-label leakage. The trial's score is the mean validation Sharpe across folds.

**What changed vs Pt2.** Pt2 ran 5 trials with no CV. Pt3 runs **150 trials** with 3-fold purged walk-forward CV. This gives the sampler enough budget to actually explore, and the fold-averaged Sharpe is a much more stable objective than a single-window estimate.

In [9]:
N_OPTUNA_TRIALS = 150

# Feature cache keyed by rsi_period
_feat_cache: dict = {}

def get_features(rsi_period: int) -> pd.DataFrame:
    if rsi_period not in _feat_cache:
        _feat_cache[rsi_period] = compute_features(prices, rsi_period=rsi_period)
    return _feat_cache[rsi_period]

def purged_walkforward_folds(unique_dates: np.ndarray,
                             train_end: pd.Timestamp,
                             val_end:   pd.Timestamp,
                             n_folds:   int = 3,
                             embargo:   int = 5) -> list:
    """Split [earliest_date, val_end) into n_folds expanding-window folds,
    each with an `embargo` trading-day gap between train-end and val-start."""
    dates_in_range = unique_dates[unique_dates < val_end]
    # Walk-forward: each fold's training runs from earliest to some cutoff,
    # validation runs on the next slice.
    # Split the train_end -> val_end validation span into n_folds chunks.
    val_dates = dates_in_range[dates_in_range >= train_end]
    if len(val_dates) < n_folds * 20:
        return []
    chunk = len(val_dates) // n_folds
    folds = []
    for i in range(n_folds):
        v_start = val_dates[i * chunk]
        v_end   = val_dates[(i+1)*chunk - 1] if i < n_folds-1 else val_dates[-1]
        # Train: everything up to v_start minus embargo
        t_end_idx = np.searchsorted(dates_in_range, v_start) - embargo
        if t_end_idx < 50:
            continue
        t_end = dates_in_range[t_end_idx]
        folds.append((t_end, v_start, v_end))
    return folds

def objective(trial: optuna.Trial) -> float:
    rsi_period    = trial.suggest_categorical("rsi_period",    [7, 14, 21])
    rebal_freq    = trial.suggest_categorical("rebal_freq",    [2, 4, 6, 8, 12])
    ridge_lambda  = trial.suggest_float(      "ridge_lambda",  0.01, 1.0, log=True)
    top_n         = trial.suggest_categorical("top_n",         [5, 7, 9, 12])
    xgb_reg_l     = trial.suggest_float(      "xgb_reg_lambda", 0.1, 10.0, log=True)
    xgb_depth     = trial.suggest_categorical("xgb_max_depth", [2, 3, 4])

    feat = get_features(rsi_period)
    folds = purged_walkforward_folds(
        np.sort(feat["date"].unique()),
        train_end=TRAIN_CUTOFF, val_end=VAL_CUTOFF, n_folds=3, embargo=5,
    )
    if not folds:
        return -99.0
    sharpes = []
    for t_end, v_start, v_end in folds:
        r = run_backtest(
            prices, feat, regimes,
            rebal_freq   = rebal_freq,
            ridge_lambda = ridge_lambda,
            top_n        = top_n,
            train_end    = pd.Timestamp(t_end),
            eval_start   = pd.Timestamp(v_start),
            eval_end     = pd.Timestamp(v_end) + pd.Timedelta(days=1),
            xgb_kwargs   = dict(reg_lambda=xgb_reg_l, max_depth=xgb_depth),
        )
        s = r["sharpe"]
        sharpes.append(s if np.isfinite(s) else -99.0)
    return float(np.mean(sharpes))

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler,
                            study_name="momentum_dow30_pt3")

print(f"\nRunning {N_OPTUNA_TRIALS}-trial Optuna TPE search with 3-fold purged walk-forward CV ...\n")
def progress_cb(study, trial):
    if trial.number % 10 == 0 or trial.number == N_OPTUNA_TRIALS - 1:
        best = study.best_value if study.best_trial else float("nan")
        v    = trial.value if trial.value is not None else float("nan")
        p    = trial.params
        print(f"  trial {trial.number+1:3d}/{N_OPTUNA_TRIALS}  "
              f"RSI={p.get('rsi_period','?'):>2}  rebal={p.get('rebal_freq','?')}w  "
              f"lam={p.get('ridge_lambda',0):.3f}  N={p.get('top_n','?'):>2}  "
              f"xgb_l={p.get('xgb_reg_lambda',0):.2f}  depth={p.get('xgb_max_depth','?')}  "
              f"CV-Sharpe={v:.3f}  (best={best:.3f})")

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[progress_cb])

best_params = study.best_params
print("\n" + "="*60)
print(f"  BEST TRIAL  #{study.best_trial.number+1}")
print("="*60)
for k,v in best_params.items():
    print(f"    {k:20s}: {v}")
print(f"    {'CV-Sharpe (mean)':20s}: {study.best_value:.4f}")


Running 150-trial Optuna TPE search with 3-fold purged walk-forward CV ...



  trial   1/150  RSI=14  rebal=12w  lam=0.159  N= 9  xgb_l=0.27  depth=4  CV-Sharpe=0.363  (best=0.363)


  trial  11/150  RSI=21  rebal=6w  lam=0.010  N= 7  xgb_l=9.27  depth=3  CV-Sharpe=1.096  (best=1.096)


  trial  21/150  RSI=21  rebal=8w  lam=0.052  N=12  xgb_l=3.18  depth=3  CV-Sharpe=1.176  (best=1.176)


  trial  31/150  RSI= 7  rebal=12w  lam=0.119  N= 5  xgb_l=1.75  depth=2  CV-Sharpe=0.737  (best=1.380)


...

  trial 111/150  RSI=21  rebal=2w  lam=0.085  N= 9  xgb_l=3.32  depth=3  CV-Sharpe=0.558  (best=1.487)


  trial 121/150  RSI=21  rebal=12w  lam=0.069  N= 5  xgb_l=1.93  depth=3  CV-Sharpe=0.931  (best=1.487)


  trial 131/150  RSI=21  rebal=8w  lam=0.040  N= 5  xgb_l=3.09  depth=3  CV-Sharpe=1.480  (best=1.487)


  trial 141/150  RSI=21  rebal=4w  lam=0.034  N= 5  xgb_l=3.44  depth=3  CV-Sharpe=0.915  (best=1.487)


  trial 150/150  RSI=21  rebal=8w  lam=0.020  N= 5  xgb_l=4.32  depth=3  CV-Sharpe=1.568  (best=1.633)

  BEST TRIAL  #144
    rsi_period          : 21
    rebal_freq          : 8
    ridge_lambda        : 0.022384415995194563
    top_n               : 5
    xgb_reg_lambda      : 2.8448257540186543
    xgb_max_depth       : 3
    CV-Sharpe (mean)    : 1.6331


## 8. Final out-of-sample test

**What this does.** Uses the best hyperparameters from the purged walk-forward search to train on 2016-04 → 2022-01 and evaluates on the held-out 2022-01 → 2026-04 test window. The train span is now larger than Pt2 (Pt2 trained only on 2016-04 → 2020-01) because the validation slice has been absorbed into the CV budget; more training data should make the ranker more stable.

**Honesty check.** If the final Pt3 Sharpe or CAGR comes in below Pt2's (0.6359 / 8.25%), Phase 4 will report the number as-is rather than cherry-picking.

In [10]:
final_feat = get_features(best_params["rsi_period"])

result = run_backtest(
    prices, final_feat, regimes,
    rebal_freq   = best_params["rebal_freq"],
    ridge_lambda = best_params["ridge_lambda"],
    top_n        = best_params["top_n"],
    train_end    = VAL_CUTOFF,          # train through end of validation → use full train+val
    eval_start   = VAL_CUTOFF,
    eval_end     = None,
    xgb_kwargs   = dict(
        reg_lambda = best_params["xgb_reg_lambda"],
        max_depth  = best_params["xgb_max_depth"],
    ),
)

print("\n" + "="*60)
print("  Pt3 FINAL OUT-OF-SAMPLE METRICS (2022-01 -> 2026-04)")
print("="*60)
print(f"  CAGR           : {result['cagr']:.2f}%  (benchmark {result['bench_cagr']:.2f}%)")
print(f"  Sharpe         : {result['sharpe']:.4f}")
print(f"  Sortino        : {result['sortino']:.4f}")
print(f"  Calmar         : {result['calmar']}")
print(f"  Info ratio     : {result['ir']:.4f}")
print(f"  Max drawdown   : {result['max_dd']:.2f}%")
print(f"  Cumulative ret : {result['cum_ret']:.1f}%")
print(f"  Hit rate       : {result['hit_rate']}")
print(f"  Avg turnover   : {result['avg_turnover']}  ({result['n_rebals']} rebalances)")
print(f"  Kill-switches  : {result['kills']}")

print("\nFeature importance (gain):")
for f, imp in sorted(result["feat_imp"].items(), key=lambda x:-x[1]):
    print(f"  {f:18s}: {imp:.4f}")


  Pt3 FINAL OUT-OF-SAMPLE METRICS (2022-01 -> 2026-04)
  CAGR           : 6.01%  (benchmark 10.15%)
  Sharpe         : 0.5030
  Sortino        : 0.6205
  Calmar         : 0.4579
  Info ratio     : -0.3615
  Max drawdown   : -13.12%
  Cumulative ret : 28.1%
  Hit rate       : 0.3704
  Avg turnover   : 1.2622  (27 rebalances)
  Kill-switches  : 4

Feature importance (gain):
  mom_3_1           : 0.1382
  ma_cross          : 0.1331
  dist_52wk_high    : 0.1288
  mom_12_1          : 0.1277
  mom_accel         : 0.1247
  mom_6_1           : 0.1198
  mom_21            : 0.1184
  rsi               : 0.1092


## 9. Year-by-year returns vs PIT benchmark and DIA ETF

**What this does.** Pulls DIA (SPDR Dow Jones Industrial Average ETF) returns from yfinance and compares the strategy's calendar-year returns against both the PIT equal-weight benchmark and the actual tradable Dow ETF.

In [11]:
strat_curve = np.array(result["strat_curve"])
bench_curve = np.array(result["bench_curve"])
dates       = pd.DatetimeIndex(result["date_index"])
rs = np.diff(strat_curve) / strat_curve[:-1]
rb = np.diff(bench_curve) / bench_curve[:-1]
df_rets = pd.DataFrame({"strat": rs, "bench_pit": rb}, index=dates[1:])

dia = yf.download("DIA", start=dates.min().date(), end=(dates.max()+pd.Timedelta(days=1)).date(),
                  auto_adjust=True, progress=False)["Close"]
dia_r = dia.pct_change().dropna()
# DIA may be a DataFrame column, squeeze to Series
if hasattr(dia_r, "squeeze"): dia_r = dia_r.squeeze()
df_rets["dia"] = dia_r

yoy = df_rets.resample("YE").apply(lambda x: (1+x).prod()-1) * 100
print("Year-by-year returns (%):")
print(yoy.round(1).to_string())

Year-by-year returns (%):
            strat  bench_pit   dia
2022-12-31   -6.0       -7.6  -8.2
2023-12-31   10.1       18.3  16.0
2024-12-31    9.4       17.1  14.8
2025-12-31   18.5       16.7  14.7
2026-12-31   -5.8        0.2   0.1


## 10. Plots & saved artefacts

**What this does.** Saves the Pt2-equivalent 6-panel backtest dashboard, the Optuna analysis chart, a `best_params.txt` text summary, and a new `pt3_vs_pt2_comparison.png` that overlays the Pt3 equity curve against Pt2's (Pt2 curve reproduced by re-running Pt2's configuration on the same feature pipeline).

In [12]:
# Plot style (same palette as Pt2 for direct visual comparison)
DARK="#0a0c0f"; SURFACE="#111418"; BORDER="#232830"; TEXT="#e2e8f0"
MUTED="#8896a8"; GREEN="#22c55e"; RED="#ef4444"; AMBER="#f59e0b"
BLUE="#60a5fa"; PURPLE="#a78bfa"

plt.rcParams.update({
    "figure.facecolor": DARK, "axes.facecolor": SURFACE,
    "axes.edgecolor": BORDER, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": TEXT, "grid.color": BORDER,
    "grid.linewidth": 0.5, "font.family": "monospace",
    "axes.titlecolor": TEXT, "axes.titlesize": 10,
    "axes.titleweight": "bold",
})

def plot_backtest(result, best_params, save_path="results/backtest_summary.png"):
    fig = plt.figure(figsize=(16,12), facecolor=DARK)
    gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35,
                            left=0.07,right=0.97,top=0.92,bottom=0.07)
    dts = pd.DatetimeIndex(result["date_index"])
    s   = np.array(result["strat_curve"]); b = np.array(result["bench_curve"])
    regs= result["regime_log"]
    rc  = {"trending":GREEN,"volatile":AMBER,"crash":RED}

    ax1 = fig.add_subplot(gs[0,:])
    ax1.plot(dts,s,color=GREEN,lw=2.0,label=f"Strategy   CAGR {result['cagr']:.1f}%",zorder=3)
    ax1.plot(dts,b,color=BLUE,lw=1.4,ls="--",alpha=0.7,
             label=f"Benchmark  CAGR {result['bench_cagr']:.1f}%",zorder=2)
    pr, ss = regs[0], dts[0]
    for i in range(1,len(dts)):
        if regs[i] != pr or i == len(dts)-1:
            ax1.axvspan(ss, dts[i], alpha=0.09, color=rc[pr], zorder=1)
            ss = dts[i]; pr = regs[i]
    legend_patches = [Patch(facecolor=GREEN,alpha=0.4,label="Trending"),
                      Patch(facecolor=AMBER,alpha=0.4,label="Volatile (70%)"),
                      Patch(facecolor=RED,  alpha=0.4,label="Crash -> cash")]
    ax1.legend(handles=[*ax1.get_lines(),*legend_patches], loc="upper left",
               fontsize=8.5, facecolor=SURFACE, edgecolor=BORDER,
               labelcolor=TEXT, ncol=2)
    ax1.set_title("Pt3  CUMULATIVE PERFORMANCE  (shaded = regime)", pad=8)
    ax1.set_ylabel("Portfolio value  (base = 100)")
    ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(gs[1,:2])
    pk = np.maximum.accumulate(s); dd = (s - pk) / pk * 100.0
    ax2.fill_between(dts, dd, 0, color=RED, alpha=0.45)
    ax2.plot(dts, dd, color=RED, lw=0.8)
    ax2.set_title(f"DRAWDOWN  (max {result['max_dd']:.1f}%)")
    ax2.set_ylabel("%"); ax2.grid(True,alpha=0.3)

    ax3 = fig.add_subplot(gs[1,2])
    fi = result["feat_imp"]
    labels = FEATURE_COLS
    vals   = [fi.get(f,0.0) for f in FEATURE_COLS]
    vp     = np.array(vals) / (sum(vals) or 1) * 100.0
    cols   = [GREEN, BLUE, PURPLE, AMBER, "#22d3ee", "#f97316", "#a3e635", RED]
    bars = ax3.barh(labels, vp, color=cols[:len(labels)], height=0.5)
    for bar, v in zip(bars, vp):
        ax3.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f"{v:.1f}%", va="center", fontsize=8, color=TEXT)
    ax3.set_title("XGBOOST FEATURE IMPORTANCE  (gain %)")
    ax3.set_xlim(0, max(vp)*1.3 if max(vp)>0 else 1)
    ax3.grid(True, alpha=0.3, axis="x")

    ax4 = fig.add_subplot(gs[2,:2])
    ri = [{"trending":1,"volatile":2,"crash":3}[r] for r in regs]
    cm = {1:GREEN,2:AMBER,3:RED}
    for i in range(len(dts)-1):
        ax4.axvspan(dts[i], dts[i+1], ymin=0, ymax=1, color=cm[ri[i]], alpha=0.75)
    ax4.set_yticks([])
    n = len(regs)
    ax4.set_title(f"REGIME TIMELINE  (kill-switch fired {result['kills']}x)")
    ax4.legend(handles=[
        Patch(color=GREEN,label=f"Trending {regs.count('trending')/n*100:.0f}%"),
        Patch(color=AMBER,label=f"Volatile {regs.count('volatile')/n*100:.0f}%"),
        Patch(color=RED,  label=f"Crash    {regs.count('crash')/n*100:.0f}%"),
    ], loc="upper right", fontsize=8, facecolor=SURFACE, edgecolor=BORDER,
       labelcolor=TEXT)

    ax5 = fig.add_subplot(gs[2,2]); ax5.axis("off")
    rows = [
        ("Ann. return",    f"{result['cagr']:.2f}%",        GREEN),
        ("Benchmark",      f"{result['bench_cagr']:.2f}%",  BLUE),
        ("Sharpe",         f"{result['sharpe']:.4f}",       GREEN if result["sharpe"]>0.5 else AMBER),
        ("Sortino",        f"{result['sortino']:.4f}",      AMBER),
        ("Calmar",         f"{result['calmar']}",           PURPLE),
        ("Info ratio",     f"{result['ir']:.4f}",           PURPLE),
        ("Max DD",         f"{result['max_dd']:.2f}%",      RED),
        ("Hit rate",       f"{result['hit_rate']}",         TEXT),
        ("Avg turnover",   f"{result['avg_turnover']}",     TEXT),
        ("Kill-switches",  f"{result['kills']}",            PURPLE),
        ("-- best params --", "", MUTED),
        ("RSI period",     f"{best_params['rsi_period']}d", TEXT),
        ("Rebal freq",     f"{best_params['rebal_freq']}w", TEXT),
        ("Ridge lambda",   f"{best_params['ridge_lambda']:.4f}", TEXT),
        ("Top-N",          f"{best_params['top_n']}",       TEXT),
    ]
    for i,(lab,val,col) in enumerate(rows):
        y = 1.0 - i*0.065
        ax5.text(0.02, y, lab, transform=ax5.transAxes, fontsize=8.5,
                 color=MUTED, va="top")
        ax5.text(0.98, y, val, transform=ax5.transAxes, fontsize=8.5,
                 color=col, va="top", ha="right", fontweight="bold")

    fig.suptitle("Pt3 DOW 30 MOMENTUM // XGBoost (8 feats) + Ridge + Calibrated Regime + 150-trial Optuna + CV",
                 fontsize=11, fontweight="bold", color=TEXT, y=0.97)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  saved: {save_path}")
    plt.close()

plot_backtest(result, best_params)

  saved: results/backtest_summary.png


In [13]:
def plot_optuna(study, save_path="results/optuna_analysis.png"):
    df = study.trials_dataframe(attrs=("number","value","params","state"))
    df = df[df["state"] == "COMPLETE"].copy()
    df.rename(columns={"value":"sharpe"}, inplace=True)
    df.sort_values("sharpe", ascending=False, inplace=True)

    fig, axes = plt.subplots(2,2, figsize=(14,10), facecolor=DARK)
    fig.suptitle("Pt3 OPTUNA SEARCH // objective: mean CV-Sharpe across 3 purged folds",
                 fontsize=11, fontweight="bold", color=TEXT, y=0.98)

    cmap = LinearSegmentedColormap.from_list("rg", [RED, AMBER, GREEN])
    vmin = df["sharpe"].quantile(0.10); vmax = df["sharpe"].quantile(0.90)

    ax = axes[0,0]
    ax.scatter(df["number"]+1, df["sharpe"], c=df["sharpe"], cmap=cmap,
               vmin=vmin, vmax=vmax, s=25, alpha=0.8, zorder=3)
    rb = df.set_index("number")["sharpe"].sort_index().cummax()
    ax.plot(rb.index+1, rb.values, color=GREEN, lw=1.5, ls="--",
            zorder=4, label="Running best")
    ax.axhline(0.5, color=AMBER, lw=0.8, ls=":", alpha=0.7)
    ax.set_xlabel("Trial"); ax.set_ylabel("Mean CV Sharpe")
    ax.set_title("OPTIMISATION HISTORY")
    ax.legend(fontsize=8, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3)

    ax = axes[0,1]
    p_rsi = "params_rsi_period"; p_rb = "params_rebal_freq"
    if p_rsi in df.columns and p_rb in df.columns:
        piv = df.groupby([p_rsi, p_rb])["sharpe"].mean().unstack(fill_value=np.nan)
        im = ax.imshow(piv.values, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels([f"{c}w" for c in piv.columns], fontsize=9)
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels(piv.index, fontsize=9)
        ax.set_xlabel("Rebalance freq"); ax.set_ylabel("RSI period")
        ax.set_title("RSI x REBAL FREQ  (mean CV Sharpe)")
        for i in range(len(piv.index)):
            for j in range(len(piv.columns)):
                v = piv.values[i,j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=7.5, color="black" if v > df["sharpe"].median() else "white")
        plt.colorbar(im, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    ax = axes[1,0]
    p_lam = "params_ridge_lambda"; p_n = "params_top_n"
    sc = ax.scatter(df["number"]+1, df["sharpe"],
                    c=df[p_lam] if p_lam in df.columns else df["sharpe"],
                    cmap=cmap, s=30, alpha=0.7, edgecolors="none")
    plt.colorbar(sc, ax=ax, label="Ridge lambda").ax.tick_params(labelcolor=MUTED)
    ax.set_xlabel("Trial"); ax.set_ylabel("CV Sharpe")
    ax.set_title("CV-SHARPE BY TRIAL  (colour = Ridge lambda)")
    ax.axhline(0.5, color=GREEN, ls="--", lw=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3)

    ax = axes[1,1]
    top20 = df.head(20).copy()
    cols = [GREEN if s>0.5 else AMBER if s>0.3 else RED for s in top20["sharpe"]]
    ax.bar(range(len(top20)), top20["sharpe"], color=cols, width=0.7)
    lbl = []
    for _, r in top20.iterrows():
        rsi = int(r.get(p_rsi, 0)); rb = int(r.get(p_rb, 0))
        lam = r.get(p_lam, 0); n = int(r.get(p_n, 0))
        lbl.append(f"R{rsi} {rb}w\nlam{lam:.2f} N{n}")
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(lbl, rotation=90, fontsize=6.5)
    ax.set_ylabel("CV Sharpe")
    ax.set_title("TOP 20 TRIALS BY CV SHARPE")
    ax.axhline(0.5, color=GREEN, ls="--", lw=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout(rect=[0,0,1,0.97])
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  saved: {save_path}")
    plt.close()

plot_optuna(study)

  saved: results/optuna_analysis.png


In [14]:
# Save best_params text summary
with open("results/best_params.txt", "w") as f:
    f.write("Pt3 BEST HYPERPARAMETERS  (Optuna TPE, purged walk-forward CV)\n")
    f.write("=" * 60 + "\n")
    for k,v in best_params.items():
        f.write(f"{k:20s}: {v}\n")
    f.write(f"\nCV-Sharpe (mean of 3 folds): {study.best_value:.4f}\n")
    f.write("\nFINAL OUT-OF-SAMPLE METRICS (2022-01 -> 2026-04)\n")
    f.write("=" * 60 + "\n")
    f.write(f"{'sharpe':20s}: {result['sharpe']:.4f}\n")
    f.write(f"{'sortino':20s}: {result['sortino']:.4f}\n")
    f.write(f"{'calmar':20s}: {result['calmar']}\n")
    f.write(f"{'info_ratio':20s}: {result['ir']:.4f}\n")
    f.write(f"{'cagr_%':20s}: {result['cagr']:.2f}\n")
    f.write(f"{'bench_cagr_%':20s}: {result['bench_cagr']:.2f}\n")
    f.write(f"{'max_drawdown_%':20s}: {result['max_dd']:.2f}\n")
    f.write(f"{'cumulative_%':20s}: {result['cum_ret']:.2f}\n")
    f.write(f"{'hit_rate':20s}: {result['hit_rate']}\n")
    f.write(f"{'avg_turnover':20s}: {result['avg_turnover']}\n")
    f.write(f"{'n_rebals':20s}: {result['n_rebals']}\n")
    f.write(f"{'kill_switches':20s}: {result['kills']}\n")
print("  saved: results/best_params.txt")

  saved: results/best_params.txt


## 11. Pt3 vs Pt2 equity-curve overlay

**What this does.** Re-runs the Pt2 configuration (3 features including `vol_60`, raw `fwd_ret` target, hardcoded regime thresholds) on the same price panel to reproduce the Pt2 equity curve, then overlays it with Pt3's curve on the same OOS window. This is the single most informative chart for "did the changes help".

In [15]:
# --- Pt2 reproduction: 3 features only, raw fwd target, Pt2 regime thresholds ---
PT2_FEATURES = ["mom_12_1", "vol_60", "rsi"]

def compute_features_pt2(prices: pd.DataFrame, rsi_period: int = 14,
                         fwd_days: int = 5) -> pd.DataFrame:
    log_ret = np.log(prices / prices.shift(1))
    mom_12_1 = prices.shift(21) / prices.shift(252) - 1.0
    vol_60   = log_ret.rolling(60).std() * np.sqrt(252)
    rsi_panel = pd.concat({c: _rsi_series(prices[c], rsi_period)
                           for c in prices.columns}, axis=1)
    fwd = prices.shift(-fwd_days) / prices - 1.0

    long = []
    for nm, df in [("mom_12_1",mom_12_1),("vol_60",vol_60),("rsi",rsi_panel)]:
        s = df.stack(future_stack=True); s.index.names = ["date","ticker"]
        long.append(s.rename(nm))
    fwd_s = fwd.stack(future_stack=True); fwd_s.index.names = ["date","ticker"]
    long.append(fwd_s.rename("fwd_ret"))
    df_long = pd.concat(long, axis=1).reset_index()

    members = {d: get_constituents_on_date(d) for d in df_long["date"].unique()}
    df_long = df_long[df_long.apply(lambda r: r["ticker"] in members[r["date"]], axis=1)]
    df_long = df_long.replace([np.inf,-np.inf], np.nan).dropna(
        subset=PT2_FEATURES + ["fwd_ret"]
    )
    df_long["fwd_rank"] = df_long.groupby("date")["fwd_ret"].rank(pct=True)
    return df_long

def run_backtest_pt2(prices, features_df, regimes_pt2,
                    rebal_freq=4, ridge_lambda=0.0124, top_n=12,
                    eval_start=VAL_CUTOFF):
    # Tiny copy of run_backtest but with PT2_FEATURES
    rebal_days = rebal_freq * 5
    unique_dates = np.sort(features_df["date"].unique())
    train_dates = unique_dates[unique_dates < TRAIN_CUTOFF]
    eval_dates  = unique_dates[unique_dates >= eval_start]
    train_df = features_df[features_df["date"].isin(train_dates)]
    clean = train_df[PT2_FEATURES + ["fwd_rank"]].replace([np.inf,-np.inf],np.nan).dropna()
    X = clean[PT2_FEATURES].values.astype(np.float32)
    y = clean["fwd_rank"].values.astype(np.float32)
    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=1.0, min_child_weight=5,
        reg_lambda=1.0, reg_alpha=0.1, objective="reg:squarederror",
        eval_metric="rmse", random_state=42, n_jobs=-1, verbosity=0,
        tree_method="hist",
    )
    model.fit(X, y)
    eval_df = features_df[features_df["date"].isin(eval_dates)]
    test_start = pd.Timestamp(eval_dates[0])
    test_prices = prices[prices.index >= test_start].copy()
    weights = {}; s_val = 100.0; b_val = 100.0
    last = -rebal_days; peak = 100.0; mdd = 0.0
    s_curve=[]; b_curve=[]; dts=[]
    for di in range(1, len(test_prices)):
        date = test_prices.index[di]
        pit = [t for t in get_constituents_on_date(date) if t in test_prices.columns]
        drets = {}
        for t in pit:
            p0 = test_prices[t].iloc[di-1]; p1 = test_prices[t].iloc[di]
            if pd.notna(p0) and pd.notna(p1) and p0>0:
                drets[t] = p1/p0 - 1.0
        reg = regimes_pt2.get(date, "trending")
        if di - last >= rebal_days:
            last = di
            avail = eval_df[eval_df["date"] <= date]
            if not avail.empty:
                sd = avail["date"].max()
                snap = (eval_df[(eval_df["date"]==sd) & (eval_df["ticker"].isin(pit))]
                        .set_index("ticker").dropna(subset=PT2_FEATURES))
                if len(snap) >= top_n:
                    sc = model.predict(snap[PT2_FEATURES].values.astype(np.float32))
                    tav = snap.index.tolist()
                    if reg == "crash":
                        weights = {}
                    else:
                        scale = 0.70 if reg=="volatile" else 1.00
                        ow = ridge_optimize(sc, tav, ridge_lambda, top_n)
                        weights = {t: w*scale for t,w in ow.items()}
        sr = sum(weights.get(t,0.0)*drets.get(t,0.0) for t in weights)
        br = float(np.mean([drets[t] for t in pit if t in drets]))              if any(t in drets for t in pit) else 0.0
        s_val *= (1+sr); b_val *= (1+br)
        peak = max(peak, s_val); mdd = min(mdd, (s_val-peak)/peak)
        s_curve.append(s_val); b_curve.append(b_val); dts.append(date)
    cagr = (s_val/100)**(252/len(s_curve))-1
    bcagr= (b_val/100)**(252/len(s_curve))-1
    rs = np.diff(s_curve)/np.array(s_curve[:-1])
    sh = (rs.mean()/rs.std())*np.sqrt(252) if rs.std()>0 else 0.0
    return dict(strat_curve=s_curve, bench_curve=b_curve, date_index=dts,
                cagr=round(cagr*100,2), bench_cagr=round(bcagr*100,2),
                sharpe=round(sh,4), max_dd=round(mdd*100,2))

# Pt2's hardcoded regime thresholds
def classify_regimes_pt2(prices):
    mkt = build_market_index(prices)
    lr = np.log(mkt/mkt.shift(1))
    rv = lr.rolling(20).std()*np.sqrt(252)
    pk = mkt.rolling(60).max()
    dd = (mkt-pk)/pk
    r = pd.Series("trending", index=prices.index)
    r[(rv > 0.17) | (dd < -0.07)] = "volatile"
    r[(rv > 0.28) | (dd < -0.13)] = "crash"
    return r

print("Reproducing Pt2 baseline on same price panel ...")
feat_pt2 = compute_features_pt2(prices, rsi_period=14)
regimes_pt2 = classify_regimes_pt2(prices)
# Use Pt2's best params: rsi=14, rebal=4w, lam=0.0124, N=12
result_pt2_repro = run_backtest_pt2(prices, feat_pt2, regimes_pt2,
                                    rebal_freq=4, ridge_lambda=0.0124, top_n=12)
print(f"  Pt2 reproduction CAGR {result_pt2_repro['cagr']:.2f}%  "
      f"Sharpe {result_pt2_repro['sharpe']:.4f}  maxDD {result_pt2_repro['max_dd']:.2f}%")

Reproducing Pt2 baseline on same price panel ...


  Pt2 reproduction CAGR 8.11%  Sharpe 0.6115  maxDD -20.81%


In [16]:
fig, ax = plt.subplots(figsize=(14,7), facecolor=DARK)
d3 = pd.DatetimeIndex(result["date_index"])
d2 = pd.DatetimeIndex(result_pt2_repro["date_index"])
ax.plot(d3, result["strat_curve"], color=GREEN, lw=2.0,
        label=f"Pt3 Strategy     CAGR {result['cagr']:.2f}%  Sharpe {result['sharpe']:.3f}")
ax.plot(d2, result_pt2_repro["strat_curve"], color=AMBER, lw=1.8, ls="--",
        label=f"Pt2 Strategy     CAGR {result_pt2_repro['cagr']:.2f}%  Sharpe {result_pt2_repro['sharpe']:.3f}")
ax.plot(d3, result["bench_curve"], color=BLUE, lw=1.2, ls=":", alpha=0.8,
        label=f"PIT Benchmark   CAGR {result['bench_cagr']:.2f}%")
ax.set_title("Pt3 vs Pt2 // Equity curves on identical OOS window",
             color=TEXT, fontsize=12, fontweight="bold")
ax.set_ylabel("Portfolio value  (base = 100)"); ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=10, facecolor=SURFACE, edgecolor=BORDER,
          labelcolor=TEXT)
plt.tight_layout()
plt.savefig("results/pt3_vs_pt2_comparison.png", dpi=150, bbox_inches="tight",
            facecolor=DARK)
print("  saved: results/pt3_vs_pt2_comparison.png")
plt.close()

  saved: results/pt3_vs_pt2_comparison.png


## 12. Summary line (honest final numbers)

In [17]:
print("="*70)
print("  Pt3 vs Pt2 — honest side-by-side")
print("="*70)
print(f"{'metric':20s}{'Pt2':>14s}{'Pt3':>14s}{'delta':>14s}")
print("-"*70)
pt2_sharpe = result_pt2_repro["sharpe"];  pt3_sharpe = result["sharpe"]
pt2_cagr   = result_pt2_repro["cagr"];    pt3_cagr   = result["cagr"]
pt2_mdd    = result_pt2_repro["max_dd"];  pt3_mdd    = result["max_dd"]
print(f"{'Sharpe':20s}{pt2_sharpe:>14.4f}{pt3_sharpe:>14.4f}{pt3_sharpe-pt2_sharpe:>+14.4f}")
print(f"{'CAGR %':20s}{pt2_cagr:>14.2f}{pt3_cagr:>14.2f}{pt3_cagr-pt2_cagr:>+14.2f}")
print(f"{'Max DD %':20s}{pt2_mdd:>14.2f}{pt3_mdd:>14.2f}{pt3_mdd-pt2_mdd:>+14.2f}")
print(f"{'Bench CAGR %':20s}{'--':>14s}{result['bench_cagr']:>14.2f}{'--':>14s}")
print("-"*70)
print(f"Pt3 extra metrics:")
print(f"  Sortino         : {result['sortino']:.4f}")
print(f"  Calmar          : {result['calmar']}")
print(f"  Info ratio      : {result['ir']:.4f}")
print(f"  Hit rate        : {result['hit_rate']}")
print(f"  Avg turnover    : {result['avg_turnover']}  across {result['n_rebals']} rebals")
print(f"  Kill-switches   : {result['kills']}")
print("="*70)

  Pt3 vs Pt2 — honest side-by-side
metric                         Pt2           Pt3         delta
----------------------------------------------------------------------
Sharpe                      0.6115        0.5030       -0.1085
CAGR %                        8.11          6.01         -2.10
Max DD %                    -20.81        -13.12         +7.69
Bench CAGR %                    --         10.15            --
----------------------------------------------------------------------
Pt3 extra metrics:
  Sortino         : 0.6205
  Calmar          : 0.4579
  Info ratio      : -0.3615
  Hit rate        : 0.3704
  Avg turnover    : 1.2622  across 27 rebals
  Kill-switches   : 4
